# 01 — Segment Discovery

## Question

What actually differentiates the supplied customer segments?

This notebook tests whether the four predefined segments are separated by
monetary behavior, transaction frequency, digital activity, or product usage.

The analysis is descriptive. The original segmentation methodology is not disclosed.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from sklearn.metrics import normalized_mutual_info_score

BASE_PATH = "/content/drive/MyDrive/PROJECTS/digital-banking-behavior/data/raw/"

customers = pd.read_csv(BASE_PATH + "customer_data.csv")

print(f"Customers: {customers.shape[0]:,}")
print(f"Columns: {customers.shape[1]}")


## 1. Segment profile

Start with the supplied labels and compare the main monetary and engagement measures.
Medians are used because several transaction variables are strongly right-skewed.


In [ ]:
segment_profile = (
    customers.groupby("customer_segment")
    .agg(
        customers=("customer_id", "size"),
        median_tx_count=("tx_count", "median"),
        median_tx_value=("avg_tx_value", "median"),
        median_total_volume=("total_tx_volume", "median"),
        median_app_logins=("app_logins_frequency", "median"),
        median_feature_diversity=("feature_usage_diversity", "median"),
        median_active_products=("active_products", "median"),
    )
    .reindex(["inactive", "occasional", "regular", "power"])
)

segment_profile.round(2)


In [ ]:
segment_mix = (
    customers["customer_segment"]
    .value_counts()
    .reindex(["inactive", "occasional", "regular", "power"])
    .to_frame("customers")
)

segment_mix["share_pct"] = segment_mix["customers"] / len(customers) * 100
segment_mix.round(2)


### Finding 1

Transaction value changes sharply across the four segments, while transaction count,
app logins, feature diversity, and active products remain nearly unchanged.

The supplied segmentation therefore appears much more closely aligned with monetary
intensity than with broader digital or product engagement.

This does not establish how the original segmentation algorithm was built.


## 2. Do value and engagement move together?

Spearman correlation is used because the variables are skewed and the question is
whether they move together monotonically, not whether they follow a linear relationship.


In [ ]:
behavior_vars = [
    "tx_count",
    "avg_tx_value",
    "total_tx_volume",
    "app_logins_frequency",
    "feature_usage_diversity",
    "active_products",
]

spearman_corr = (
    customers[behavior_vars]
    .corr(method="spearman")
    .round(3)
)

spearman_corr


In [ ]:
value_engagement_corr = (
    spearman_corr.loc[
        ["app_logins_frequency", "feature_usage_diversity", "active_products"],
        ["avg_tx_value"]
    ]
    .rename(columns={"avg_tx_value": "spearman_rho"})
)

value_engagement_corr


### Finding 2

The observed associations between average transaction value and the main engagement
signals are effectively near zero.

In this dataset, monetary value and engagement behave like separate dimensions.

Because many conceptually related variables are also weakly associated, these patterns
should be treated as properties of this dataset rather than generalized to real-world
fintech customers.


## 3. Does engagement variation still exist?

Flat segment medians do not mean every customer behaves the same. The next checks look
at the overall distributions and then compare engagement ranges within each segment.


In [ ]:
distribution_summary = customers[
    [
        "tx_count",
        "avg_tx_value",
        "app_logins_frequency",
        "feature_usage_diversity",
        "active_products",
    ]
].quantile([0, .10, .25, .50, .75, .90, .95, .99, 1]).T

distribution_summary.round(2)


In [ ]:
within_segment = (
    customers.groupby("customer_segment")
    .agg(
        app_logins_p25=("app_logins_frequency", lambda x: x.quantile(.25)),
        app_logins_median=("app_logins_frequency", "median"),
        app_logins_p75=("app_logins_frequency", lambda x: x.quantile(.75)),
        features_p25=("feature_usage_diversity", lambda x: x.quantile(.25)),
        features_median=("feature_usage_diversity", "median"),
        features_p75=("feature_usage_diversity", lambda x: x.quantile(.75)),
        products_p25=("active_products", lambda x: x.quantile(.25)),
        products_median=("active_products", "median"),
        products_p75=("active_products", lambda x: x.quantile(.75)),
    )
    .reindex(["inactive", "occasional", "regular", "power"])
)

within_segment


### Finding 3

Engagement varies across customers, but the predefined segments do not organize that
variation well. Customers within the same segment can still differ in app activity,
feature usage, and product breadth.

This is why the analysis does not create a single engagement score: the components are
weakly related and there is no defensible weighting scheme.


## 4. How closely do segments align with transaction value?

Customers are divided into transaction-value deciles to test whether the segment pattern
holds across the full distribution rather than only at the four segment medians.


In [ ]:
customers["value_decile"] = pd.qcut(
    customers["avg_tx_value"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1

value_segment_share = (
    pd.crosstab(
        customers["value_decile"],
        customers["customer_segment"],
        normalize="index"
    )
    .mul(100)
    .reindex(columns=["inactive", "occasional", "regular", "power"])
)

value_segment_share.round(2)


In [ ]:
nmi = normalized_mutual_info_score(
    customers["customer_segment"],
    customers["value_decile"]
)

print(f"NMI between customer segment and transaction-value decile: {nmi:.2f}")


## Conclusion

The supplied customer segments align closely with monetary transaction value, while
digital and product engagement remain largely separate.

The evidence supports using the existing segments as a value-oriented lens, but not as
a complete description of customer relationship depth.

### What this analysis does not prove

- The original segmentation algorithm is unknown.
- Transaction value cannot be claimed as the direct cause of segment membership.
- Near-zero engagement relationships should not be generalized beyond this dataset.
